In [1]:
import pandas as pd
import os

In [2]:
all_tariffs = pd.read_csv('all_tariffs.csv')
years = all_tariffs['year'].unique()
all_tariffs 

/var/folders/87/vb506yp95bn1dt3x5p9sbk4r0000gn/T/ipykernel_4375/1533025050.py:1: DtypeWarning: Columns (1,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,26) have mixed types. Specify dtype option on import or set low_memory=False.
  all_tariffs = pd.read_csv('all_tariffs.csv')


,year,hs,brief_description,mfn_ad_val_rate,mfn_specific_rate,unit_1,unit_2,quantity_1_code,quantity_2_code,description_1,...,country_1,country_duty_1,country_2,country_duty_2,country_3,country_duty_3,country_desc,country_codes,mfn_other_rate,mfn_rate_type_code
0,1789,NaN,distilled spirits of Jamaica proof,0.0,10,per gallon,NaN,NaN,NaN,imported from any kingdom of country whatsoever,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1789,NaN,other distilled spirits,0.0,8,per gallon,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1789,NaN,molasses,0.0,2.5,per gallon,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1789,NaN,Madeira wine,0.0,18,per gallon,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1789,NaN,all other wines,0.0,10,per gallon,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
758605,2023,99990061,Imports of wool apparel from Mexico described ...,NaN,NaN,NaN,NaN,SME,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
758606,2023,99990062,Imports of textile and apparel goods from Mexi...,NaN,NaN,NaN,NaN,SME,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
758607,2023,99990064,Imports of textile and apparel goods from Mexi...,NaN,NaN,NaN,NaN,SME,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
758608,2023,99990084,Goods imported from Singapore and treated as o...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N


N+1 -> N

In [3]:
# Directory containing the mapping files
mapping_dir = "./N1_N_mappings"

# Generate all possible filename pairs for consecutive years
mapping_dfs = []
for i in range(len(years)-1, 0, -1):
    y1, y2 = years[i], years[i-1]
    print(f"Processing mapping for years: {y1} to {y2}")
    filename = f"mapped_{y1}_{y2}.csv"
    filepath = os.path.join(mapping_dir, filename)
    if os.path.exists(filepath):
        df_map = pd.read_csv(filepath)
        df_map['from_year'] = y1
        df_map['to_year'] = y2
        df_map["year_N"] = y2
        df_map["year_N1"] = y1
        mapping_dfs.append(df_map)
    else:
        print(f"File {filename} does not exist in the directory {mapping_dir}. Skipping.")

# Concatenate all mapping dataframes
all_mappings_1 = pd.concat(mapping_dfs, ignore_index=True)

Processing mapping for years: 2023 to 2022
Processing mapping for years: 2022 to 2021
Processing mapping for years: 2021 to 2020
Processing mapping for years: 2020 to 2019
Processing mapping for years: 2019 to 2018
Processing mapping for years: 2018 to 2017
Processing mapping for years: 2017 to 2016
Processing mapping for years: 2016 to 2015
Processing mapping for years: 2015 to 2014
Processing mapping for years: 2014 to 2013
Processing mapping for years: 2013 to 2012
Processing mapping for years: 2012 to 2011
Processing mapping for years: 2011 to 2010
Processing mapping for years: 2010 to 2009
Processing mapping for years: 2009 to 2008
Processing mapping for years: 2008 to 2007
Processing mapping for years: 2007 to 2006
Processing mapping for years: 2006 to 2005
Processing mapping for years: 2005 to 2004
Processing mapping for years: 2004 to 2003
Processing mapping for years: 2003 to 2002
Processing mapping for years: 2002 to 2001
Processing mapping for years: 2001 to 2000
Processing 

In [4]:
def clean_hs_codes(df, year_col='from_year', code_col='HS_N1', year_threshold=1989):
    # Only modify rows where from_year >= year_threshold and HS_N1 is not null
    mask = (df[year_col] >= year_threshold) 
    # Convert to string
    df.loc[mask, code_col] = df.loc[mask, code_col].astype(str)
    # Remove trailing '.0' if present and length > 8
    df.loc[mask, code_col] = df.loc[mask, code_col].apply(lambda x: x[:-2] if len(x) > 8 and x.endswith('.0') else x)
    # Remove rows with 10-digit codes
    df = df[~((df[year_col] >= year_threshold) & (df[code_col].astype(str).str.len() >= 10))]
    # Add leading zero to 7-digit codes
    mask = (df[year_col] >= year_threshold) & (df[code_col].astype(str).str.len() == 7)
    df.loc[mask, code_col] = df.loc[mask, code_col].apply(lambda x: '0' + x)
    return df

all_mappings_1 = clean_hs_codes(all_mappings_1)

In [5]:
df_1989_onwards = all_mappings_1[all_mappings_1['from_year'] >= 1989]

df_1989_onwards["HS_N1"].astype(str).str.len().value_counts()

HS_N1
8    472869
Name: count, dtype: int64

In [6]:
# i want to plot the average combined_similarity for each year in from_year


In [7]:
import matplotlib.pyplot as plt
import plotly.express as px

avg_similarity_by_year = all_mappings_1.groupby('from_year')['Combined_Similarity'].mean()

fig = px.line(
    avg_similarity_by_year,
    labels={'index': 'from_year', 'value': 'Average Combined Similarity'},
    title='Average Combined Similarity by from_year'
)
fig.update_traces(mode='lines+markers')
fig.update_layout(xaxis_title='from_year', yaxis_title='Average Combined Similarity')
fig.show()


In [8]:
all_mappings_1



,HS_N1,Description_N1,Mapped_HS,Mapped_Description,Cosine_Similarity,Jaccard_Similarity,Levenshtein_Similarity,Combined_Similarity,from_year,to_year,year_N,year_N1
0,01013000,Live asses,1013000,Live asses,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023
1,01031000,Live purebred breeding swine,1031000,Live purebred breeding swine,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023
2,01022920,Cows imported specially for dairy purposes,1022920,Cows imported specially for dairy purposes,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023
3,01019040,Mules and hinnies not imported for immediate s...,1019040,Mules and hinnies not imported for immediate s...,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023
4,01012900,Live horses other than purebred breeding horses,1012900,Live horses other than purebred breeding horses,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023
...,...,...,...,...,...,...,...,...,...,...,...,...
758523,NaN,"canes, walking sticks and whips",NaN,"canes, walking sticks and whips",1.000000,1.000000,1.000000,1.000000,1790,1789,1789,1790
758524,NaN,"anchors, all wares of tin, pewter, or copper, ...",NaN,"anchors, and wrought, tin, and pewter ware,",0.846574,0.142857,0.560748,0.677248,1790,1789,1789,1790
758525,NaN,medicinal drugs,NaN,manufactured tobacco,0.329250,0.000000,0.228571,0.253332,1790,1789,1789,1790
758526,NaN,"All coaches, chariots, phaetons, chaises, chai...",NaN,"every coach, chariot or other four wheel carri...",0.836723,0.120000,0.594340,0.669140,1790,1789,1789,1790


In [9]:
def clean_hs_codes2(df, year_col='to_year', code_col='Mapped_HS', year_threshold=1989):
    # Only modify rows where from_year >= year_threshold and HS_N1 is not null
    mask = (df[year_col] >= year_threshold) 
    # Convert to string
    df.loc[mask, code_col] = df.loc[mask, code_col].astype(str)
    # Remove trailing '.0' if present and length > 8
    df.loc[mask, code_col] = df.loc[mask, code_col].apply(lambda x: x[:-2] if len(x) > 8 and x.endswith('.0') else x)
    # Remove rows with 10-digit codes
    df = df[~((df[year_col] >= year_threshold) & (df[code_col].astype(str).str.len() >= 10))]
    # Add leading zero to 7-digit codes
    mask = (df[year_col] >= year_threshold) & (df[code_col].astype(str).str.len() == 7)
    df.loc[mask, code_col] = df.loc[mask, code_col].apply(lambda x: '0' + x)
    return df


all_mappings_1 = clean_hs_codes2(all_mappings_1)

In [10]:
all_mappings_1

,HS_N1,Description_N1,Mapped_HS,Mapped_Description,Cosine_Similarity,Jaccard_Similarity,Levenshtein_Similarity,Combined_Similarity,from_year,to_year,year_N,year_N1
0,01013000,Live asses,01013000,Live asses,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023
1,01031000,Live purebred breeding swine,01031000,Live purebred breeding swine,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023
2,01022920,Cows imported specially for dairy purposes,01022920,Cows imported specially for dairy purposes,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023
3,01019040,Mules and hinnies not imported for immediate s...,01019040,Mules and hinnies not imported for immediate s...,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023
4,01012900,Live horses other than purebred breeding horses,01012900,Live horses other than purebred breeding horses,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023
...,...,...,...,...,...,...,...,...,...,...,...,...
758523,NaN,"canes, walking sticks and whips",NaN,"canes, walking sticks and whips",1.000000,1.000000,1.000000,1.000000,1790,1789,1789,1790
758524,NaN,"anchors, all wares of tin, pewter, or copper, ...",NaN,"anchors, and wrought, tin, and pewter ware,",0.846574,0.142857,0.560748,0.677248,1790,1789,1789,1790
758525,NaN,medicinal drugs,NaN,manufactured tobacco,0.329250,0.000000,0.228571,0.253332,1790,1789,1789,1790
758526,NaN,"All coaches, chariots, phaetons, chaises, chai...",NaN,"every coach, chariot or other four wheel carri...",0.836723,0.120000,0.594340,0.669140,1790,1789,1789,1790


In [11]:
# make sure all years are present in the mapping

# add columns to all_mappings_1 where new column 'HS_N' is the same as 'Mapped_HS'
all_mappings_1['HS_N'] = all_mappings_1['Mapped_HS']
all_mappings_1

,HS_N1,Description_N1,Mapped_HS,Mapped_Description,Cosine_Similarity,Jaccard_Similarity,Levenshtein_Similarity,Combined_Similarity,from_year,to_year,year_N,year_N1,HS_N
0,01013000,Live asses,01013000,Live asses,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023,01013000
1,01031000,Live purebred breeding swine,01031000,Live purebred breeding swine,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023,01031000
2,01022920,Cows imported specially for dairy purposes,01022920,Cows imported specially for dairy purposes,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023,01022920
3,01019040,Mules and hinnies not imported for immediate s...,01019040,Mules and hinnies not imported for immediate s...,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023,01019040
4,01012900,Live horses other than purebred breeding horses,01012900,Live horses other than purebred breeding horses,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023,01012900
...,...,...,...,...,...,...,...,...,...,...,...,...,...
758523,NaN,"canes, walking sticks and whips",NaN,"canes, walking sticks and whips",1.000000,1.000000,1.000000,1.000000,1790,1789,1789,1790,NaN
758524,NaN,"anchors, all wares of tin, pewter, or copper, ...",NaN,"anchors, and wrought, tin, and pewter ware,",0.846574,0.142857,0.560748,0.677248,1790,1789,1789,1790,NaN
758525,NaN,medicinal drugs,NaN,manufactured tobacco,0.329250,0.000000,0.228571,0.253332,1790,1789,1789,1790,NaN
758526,NaN,"All coaches, chariots, phaetons, chaises, chai...",NaN,"every coach, chariot or other four wheel carri...",0.836723,0.120000,0.594340,0.669140,1790,1789,1789,1790,NaN


In [12]:
type(all_mappings_1["HS_N1"].iloc[0])

# make sure all HS_N1 are strings and all HS_N are strings
all_mappings_1["HS_N1"] = all_mappings_1["HS_N1"].astype(str)
all_mappings_1["HS_N"] = all_mappings_1["HS_N"].astype(str)

In [13]:
all_mappings_1

,HS_N1,Description_N1,Mapped_HS,Mapped_Description,Cosine_Similarity,Jaccard_Similarity,Levenshtein_Similarity,Combined_Similarity,from_year,to_year,year_N,year_N1,HS_N
0,01013000,Live asses,01013000,Live asses,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023,01013000
1,01031000,Live purebred breeding swine,01031000,Live purebred breeding swine,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023,01031000
2,01022920,Cows imported specially for dairy purposes,01022920,Cows imported specially for dairy purposes,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023,01022920
3,01019040,Mules and hinnies not imported for immediate s...,01019040,Mules and hinnies not imported for immediate s...,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023,01019040
4,01012900,Live horses other than purebred breeding horses,01012900,Live horses other than purebred breeding horses,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023,01012900
...,...,...,...,...,...,...,...,...,...,...,...,...,...
758523,nan,"canes, walking sticks and whips",NaN,"canes, walking sticks and whips",1.000000,1.000000,1.000000,1.000000,1790,1789,1789,1790,nan
758524,nan,"anchors, all wares of tin, pewter, or copper, ...",NaN,"anchors, and wrought, tin, and pewter ware,",0.846574,0.142857,0.560748,0.677248,1790,1789,1789,1790,nan
758525,nan,medicinal drugs,NaN,manufactured tobacco,0.329250,0.000000,0.228571,0.253332,1790,1789,1789,1790,nan
758526,nan,"All coaches, chariots, phaetons, chaises, chai...",NaN,"every coach, chariot or other four wheel carri...",0.836723,0.120000,0.594340,0.669140,1790,1789,1789,1790,nan


In [14]:
df_1989_onwards = all_mappings_1[all_mappings_1['from_year'] >= 1989]

df_1989_onwards["HS_N1"].astype(str).str.len().value_counts()

HS_N1
8    472869
Name: count, dtype: int64

In [15]:
df_1989_onwards = all_mappings_1[all_mappings_1['to_year'] >= 1989]

df_1989_onwards["HS_N"].astype(str).str.len().value_counts()

HS_N
8    463834
Name: count, dtype: int64

In [16]:
all_mappings_1

,HS_N1,Description_N1,Mapped_HS,Mapped_Description,Cosine_Similarity,Jaccard_Similarity,Levenshtein_Similarity,Combined_Similarity,from_year,to_year,year_N,year_N1,HS_N
0,01013000,Live asses,01013000,Live asses,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023,01013000
1,01031000,Live purebred breeding swine,01031000,Live purebred breeding swine,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023,01031000
2,01022920,Cows imported specially for dairy purposes,01022920,Cows imported specially for dairy purposes,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023,01022920
3,01019040,Mules and hinnies not imported for immediate s...,01019040,Mules and hinnies not imported for immediate s...,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023,01019040
4,01012900,Live horses other than purebred breeding horses,01012900,Live horses other than purebred breeding horses,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023,01012900
...,...,...,...,...,...,...,...,...,...,...,...,...,...
758523,nan,"canes, walking sticks and whips",NaN,"canes, walking sticks and whips",1.000000,1.000000,1.000000,1.000000,1790,1789,1789,1790,nan
758524,nan,"anchors, all wares of tin, pewter, or copper, ...",NaN,"anchors, and wrought, tin, and pewter ware,",0.846574,0.142857,0.560748,0.677248,1790,1789,1789,1790,nan
758525,nan,medicinal drugs,NaN,manufactured tobacco,0.329250,0.000000,0.228571,0.253332,1790,1789,1789,1790,nan
758526,nan,"All coaches, chariots, phaetons, chaises, chai...",NaN,"every coach, chariot or other four wheel carri...",0.836723,0.120000,0.594340,0.669140,1790,1789,1789,1790,nan


In [17]:
# save the dataframe to a pickle file called mapping_N1_N
all_mappings_1.to_pickle('mapping_N1_N.pkl')

In [18]:
# open the pickle file
df = pd.read_pickle("mapping_N1_N.pkl")
df

,HS_N1,Description_N1,Mapped_HS,Mapped_Description,Cosine_Similarity,Jaccard_Similarity,Levenshtein_Similarity,Combined_Similarity,from_year,to_year,year_N,year_N1,HS_N
0,01013000,Live asses,01013000,Live asses,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023,01013000
1,01031000,Live purebred breeding swine,01031000,Live purebred breeding swine,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023,01031000
2,01022920,Cows imported specially for dairy purposes,01022920,Cows imported specially for dairy purposes,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023,01022920
3,01019040,Mules and hinnies not imported for immediate s...,01019040,Mules and hinnies not imported for immediate s...,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023,01019040
4,01012900,Live horses other than purebred breeding horses,01012900,Live horses other than purebred breeding horses,1.000000,1.000000,1.000000,1.000000,2023,2022,2022,2023,01012900
...,...,...,...,...,...,...,...,...,...,...,...,...,...
758523,nan,"canes, walking sticks and whips",NaN,"canes, walking sticks and whips",1.000000,1.000000,1.000000,1.000000,1790,1789,1789,1790,nan
758524,nan,"anchors, all wares of tin, pewter, or copper, ...",NaN,"anchors, and wrought, tin, and pewter ware,",0.846574,0.142857,0.560748,0.677248,1790,1789,1789,1790,nan
758525,nan,medicinal drugs,NaN,manufactured tobacco,0.329250,0.000000,0.228571,0.253332,1790,1789,1789,1790,nan
758526,nan,"All coaches, chariots, phaetons, chaises, chai...",NaN,"every coach, chariot or other four wheel carri...",0.836723,0.120000,0.594340,0.669140,1790,1789,1789,1790,nan


In [ ]:
df_1989_onwards = df[df['from_year'] >= 1989]

df_1989_onwards["HS_N1"].astype(str).str.len().value_counts()

HS_N1
8    472869
Name: count, dtype: int64

In [20]:
df_1989_onwards = df[df['to_year'] >= 1989]

df_1989_onwards["HS_N"].astype(str).str.len().value_counts()

HS_N
8    463834
Name: count, dtype: int64